In [17]:
from enum import IntEnum
from typing import Union

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field, computed_field
from rich import print as rprint
from config.config import ToolsModel

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body = {
        "thinking":{"type":"disabled"}
    }
)

In [13]:
class UserRole(IntEnum):
    GUEST = (0, "游客", "guest")
    USER = (1, "普通用户", "user")
    ADMIN = (2, "管理员", "admin")
    SUPER_ADMIN = (3, "超级管理员", "super_admin")

    def __new__(cls, code: int, label: str, permission: str):
        obj = int.__new__(cls, code)
        obj._value_ = code
        obj.label = label
        obj.permission = permission
        return obj

In [14]:
class ContactInfo(BaseModel):
    """用户联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱")
    phone: str = Field(description="用户电话号")

class UserInfo(BaseModel):
    """用户身份信息"""
    name: str = Field(description="用户姓名")
    age: int = Field(description="用户年龄")
    role: UserRole = Field(description="用户身份")

    @computed_field
    @property
    def role_label(self) -> str:
        return self.role.label

    @computed_field
    @property
    def role_permission(self) -> str:
        return self.role.permission


# Agent的高级用法-ToolStrategy

## 1 ToolStrategy的多种输出方式：schema参数

### 1.1 Pydantic类型

举例1：

In [ ]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

resp = agent.invoke({
    "messages":[
        HumanMessage(content="小明的邮箱是spf20031014@163.com,电话是18242141179")
    ]
})

rprint(resp)


## 2 自定义工具信息： tool_message_content参数

### 用于简化工具消息，减少token，不影响后续功能

举例：

In [ ]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=ContactInfo,
        tool_message_content="已成功格式化返回数据"
    )
)

resp = agent.invoke({
    "messages":[
        HumanMessage(content="小明的邮箱是spf20031014@163.com,电话是18242141179")
    ]
})

rprint(resp)

## 3 错误处理: handle_errors参数

### 3.1 默认方式 bool（默认为true 开启机制，捕获所有异常，使用内置的错误消息模板提示模型重试）

举例1：

In [37]:
agent = create_agent(
    model=model,
    name="agent01",
    response_format=ToolStrategy(
        Union[UserInfo,ContactInfo],
        tool_message_content="已成功格式化返回数据",
        handle_errors=True
    )
)

try:
    resp = agent.invoke({
        "messages":[
            HumanMessage(content="小明邮箱是spf20031014@163.com,电话是18242141179"),
            HumanMessage(content="小明今年23岁，身份权限是管理员")
        ]
    })
except Exception as e:
    raise Exception(f"调用agent01发生错误:{e}")

if isinstance(resp["structured_response"],UserInfo):
    rprint(resp["structured_response"].model_dump_json())
else:
    rprint(resp["structured_response"])

{"name":"小明","age":23,"role":2,"role_label":"管理员","role_permission":"admin"}

举例2：
handle_errors=False(关闭重试机制，任何异常都会直接抛出，并中断程序)

In [24]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[UserInfo,ContactInfo],
        tool_message_content="已成功格式化返回数据",
        handle_errors=False
    )
)

try:
    resp = agent.invoke({
        "messages":[
            HumanMessage(content="小明今年23岁，邮箱是spf20031014@163.com,电话是18242141179，身份权限是管理员"),
            HumanMessage(content="小明今年23岁，身份权限是管理员")
        ]
    })
except Exception as e:
    raise Exception(f"格式化错误：{e}")

rprint(resp)

Exception: 发生错误：Model incorrectly returned multiple structured responses (UserInfo, ContactInfo) when only one is expected.

### 3.2 设置为固定字符串

举例1：

In [ ]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[UserInfo,ContactInfo],
        tool_message_content="已成功格式化返回数据",
        handle_errors="请检查输入的数据"
    )
)

try:
    resp = agent.invoke({
        "messages":[
            HumanMessage(content="小明邮箱是spf20031014@163.com,电话是18242141179"),
            HumanMessage(content="小明今年23岁，身份权限是管理员")
        ]
    })
except Exception as e:
    raise RuntimeError("调用 agent01 失败") from e

if isinstance(resp["structured_response"],UserInfo):
    rprint(resp["structured_response"].model_dump_json())
else:
    rprint(resp["structured_response"])

### 3.2 设置为指定异常类型

- MultipleStructuredOutputsError: 多结构化输出错误
- StructuredOutputValidationError: 输出结构化验证错误(已废弃)

举例：

In [ ]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[UserInfo,ContactInfo],
        tool_message_content="已成功格式化返回数据",
        handle_errors=(MultipleStructuredOutputsError)
    )
)

try:
    resp = agent.invoke({
        "messages":[
            HumanMessage(content="小明邮箱是spf20031014@163.com,电话是18242141179"),
            HumanMessage(content="小明今年23岁，身份权限是管理员")
        ]
    })
except Exception as e:
    raise RuntimeError("调用 agent01 失败") from e

if isinstance(resp["structured_response"],UserInfo):
    rprint(resp["structured_response"].model_dump_json())
else:
    rprint(resp["structured_response"])